# Problem 031:  Coin Sums

> In the United Kingdom the currency is made up of pound (£) and pence (p). There are eight coins in general circulation:
>
> >    1p, 2p, 5p, 10p, 20p, 50p, £1 (100p), and £2 (200p).
>
> It is possible to make £2 in the following way:
>
> >    1×£1 + 1×50p + 2×20p + 1×5p + 1×2p + 3×1p
>
> How many different ways can £2 be made using any number of coins?

## Recursive solution $\mathcal O(NM)$

One approach to this problem is to break it down recursively.  Build a valid pile by only adding coins smaller or equal to than the last coin added.  Then the number of ways of building a total of $N$ given $M$ coins will be equal to the sum of ways of building $N-c_i$ from $M - i$ for all coins $c_i$, $0$-indexed.  The terminating conditions are when the sum is equal $0$ - which has a count of $1$, or less than zero, which is impossible and has a count of $0$.

I place the routine `count_ways` to make the list `coins` accessible to the subroutine without passing it explicitly and to relieve memory storage on the recursive algorithm.   Furthermore, the cache requires cache-able input paramets, so passing a list would not work.

The use of a cache is important here; the subroutine is called many times with the exact same inputs.  On my machine it is a factor of $1,000$ times faster to run with the cache than without.  I say the time scaling is $\mathcal O(NM)$ where $N$ is the target value and $M$ is the number of denominations because the search tries to build up every combination of coins.

In [60]:
%%time

from functools import cache

def p031_recursive(N: int, coins: list[int]) -> int:
    @cache
    def count_ways(n: int, nc: int) -> int:
        if n == 0:
            return 1
        if n < 0:
            return 0
        return sum(count_ways(n - c, i + 1) for i, c in enumerate(coins[:nc]))
    
    return count_ways(N, len(coins))

N = 200
coins = [1, 2, 5, 10, 20, 50, 100, 200]
ans = p031_recursive(N, coins)

print(ans)

73682
CPU times: user 3.57 ms, sys: 4 μs, total: 3.58 ms
Wall time: 3.5 ms


## Dynamic Programming solution $\mathcal O(NM)$

Any code that can be written recursively can be written linearly.  Doesn't mean it should, but it's always a possibility.  I gave myself the challenge of writing the same algorithm but linearly, with and without memoization.  I guess it isn't exactly the same in that it goes through the denominations in the opposite order than the above (only add coins larger than the current denomination, as opposed to smaller), but the idea remains the same.

I quickly wrote the code that forgoes the memoization and has comparable timing as the cahche-less code above, and include it here due to its relative simplicity.  I will admit that I struggled in comparison to get the memoization into the code, and it's a lot more clunky.  But by hook or by crook, it has similar timing as the cached example above, outperforming the non-cached version by a factor of several $100$.

In [48]:
%%time

def p031_dp(N: int, coins: list[int]) -> int:
    total = 0

    nc = len(coins)
    
    used = [None] * (N+2)
    used[0] = 0
    current = coins[0]
    n = 1

    while used[0] is not None:
        #print(used[:n+1], current)
        if current >= N:
            if current == N:
                total += 1
            current += -coins[used[n-1]] + coins[-1]
            used[n-1] = nc-1
            
            while n > 0 and used[n-1] == nc - 1:
                n -= 1
                current -= coins[used[n]]
                used[n] = None
            if n != 0:
                current -= coins[used[n-1]]
                used[n-1] += 1
                current += coins[used[n-1]]
        else:
            used[n] = used[n-1]
            current += coins[used[n]]
            n += 1
    
    return total

N = 200
coins = [1, 2, 5, 10, 20, 50, 100, 200]
ans = p031_dp(N, coins)

print(ans)

73682
CPU times: user 2.14 s, sys: 2.51 ms, total: 2.14 s
Wall time: 2.15 s


In [49]:
%%time

def p031_dpm(N: int, coins: list[int]) -> int:
    total = 0

    
    nc = len(coins)
    
    used = [None] * (N+2)
    used[0] = 0
    current = coins[0]
    n = 1

    memo = [[None for j in range(nc)] for i in range(N)]
    
    while used[0] is not None:
        #print(used[:n+1], current)
        if current >= N or memo[current][used[n-1]] is not None:
            if current == N:
                val = 1
                current += -coins[used[n-1]] + coins[-1]
                used[n-1] = nc-1
            elif current > N:
                val = 0
                current += -coins[used[n-1]] + coins[-1]
                used[n-1] = nc-1
            else:
                val = memo[current][used[n-1]]
            
            while n > 0 and used[n-1] == nc - 1:
                n -= 1
                current -= coins[used[n]]
                used[n] = None
                if n != 0:
                    val += memo[current][used[n-1]]
                    memo[current][used[n-1]] = val
            if n != 0:
                current -= coins[used[n-1]]
                if n != 1:
                    val += memo[current][used[n-2]]
                    memo[current][used[n-2]] = val
                else:
                    total += val
                used[n-1] += 1
                current += coins[used[n-1]]
            else:
                total += val
        else:
            memo[current][used[n-1]] = 0
            used[n] = used[n-1]
            current += coins[used[n]]
            n += 1
    
    return total

N = 200
coins = [1, 2, 5, 10, 20, 50, 100, 200]
ans = p031_dpm(N, coins)

print(ans)

73682
CPU times: user 4.34 ms, sys: 0 ns, total: 4.34 ms
Wall time: 4.37 ms


##  Table solution $\mathcal O(NM)$

An even faster approach than the above is to build the answer by induction.  Notice that in the above code, several trial steps will make a sum that overshoots the goal.  These steps are avoided by considering how to build a collection of value $n$ and using the first $m$ coins in the list given the result of every $N$ with lesser values.  Namely
$$A_{n, m} = A_{n, m-1} + A_{n - c_m, m},$$
where $A_{N,M}$ is the count, and it is defined to be $0$ for any $N < 0$ or $M < 0$.  Starting with the fact that there is only one way to make $0$ with any collection of coins, the answer is built up to $A_{N,M}$.

This still requires $\mathcal O(NM)$ time complexity, as the entire $N\times M$ list is being populated, but still outperforms the two approaches given above.

In [59]:
%%time

def p031_table(N: int, coins: list[int]) -> int:
    nc = len(coins)
    
    tab = [[1 for j in range(nc)]] + [None for i in range(N)]

    for i in range(1, N+1):
        s = 0
        tab[i] = [s := s + (tab[i-coins[j]][j] if i - coins[j] >= 0 else 0) for j in range(nc)]
        
    return tab[N][-1]

N = 200
coins = [1, 2, 5, 10, 20, 50, 100, 200]
ans = p031_table(N, coins)

print(ans)

73682
CPU times: user 308 μs, sys: 190 μs, total: 498 μs
Wall time: 498 μs
